<a href="https://colab.research.google.com/github/CPTR295/NLP-Basic-Using-Pytorch/blob/main/PreTrained_Embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import torch
import torch.nn as nn
from tqdm import tqdm
!pip install annoy
from annoy import AnnoyIndex #Used for fast high dimension vector search
import numpy as np


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 29.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp312-cp312-linux_x86_64.whl size=551809 sha256=fd994a32a541c74867f55a7cdd54fa1c1afd418f6a988d33a3176098e56271a6
  Stored in directory: /root/.cache/pip/wheels/db/b9/53/a3b2d1fe1743abadddec6aa541294b24fdbc39d7800bc57311
Successfully built annoy


In [11]:
!wget https://nlp.stanford.edu/data/glove.6B.zip
#Stanford Embedding

--2026-07-12 03:56:15--  https://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-07-12 03:56:15--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  5.09MB/s    in 2m 50s  

2026-07-12 03:59:06 (4.85 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]



In [12]:
!unzip glove.6B.zip

Archive:  glove.6B.zip
  inflating: glove.6B.50d.txt        
  inflating: glove.6B.100d.txt       
  inflating: glove.6B.200d.txt       
  inflating: glove.6B.300d.txt       


In [15]:
class PreTrainedEmbedding(object):
  #A Wrapper around pretrained embedding
  def __init__(self,word_to_index,word_vectors):
    self.word_to_index = word_to_index
    self.word_vectors = word_vectors
    self.index_to_word = {v:k for k,v in self.word_to_index.items()}
    self.index = AnnoyIndex(len(word_vectors[0]),metric='euclidean')
    print('Building Index')
    for _,i in self.word_to_index.items():
      self.index.add_item(i,self.word_vectors[i])
    self.index.build(50)
    print('Finished')

  @classmethod
  def from_embeddings_file(cls,embedding_file):
    word_to_index ={}
    word_vectors =[]

    with open(embedding_file) as fp:
      for line in fp.readlines():
        line = line.split(" ")
        word = line[0]
        vec = np.array([float(x) for x in line[1:]])
        word_to_index[word]=len(word_to_index)
        word_vectors.append(vec)
    return cls(word_to_index,word_vectors)

  def get_embedding(self,word):
    return self.word_vectors[self.word_to_index[word]]

  def get_closest_to_vector(self,vector,n=1):
    nn_indices = self.index.get_nns_by_vector(vector,n)
    return [self.index_to_word[neighbor] for neighbor in nn_indices]

  def compute_and_print_analogy(self,word1,word2,word3):
    vec1 = self.get_embedding(word1)
    vec2 = self.get_embedding(word2)
    vec3 = self.get_embedding(word3)

    spatial_relationship = vec2-vec1
    vec4 = vec3+spatial_relationship

    closest_words = self.get_closest_to_vector(vec4,n=4)
    existing_words = set([word1,word2,word3])
    closest_words = [word for word in closest_words if word not in existing_words]

    if(len(closest_words)==0):
      print('Could not find nearest neighbors')
      return

    for word4 in closest_words:
      print('{} : {} :: {} : {}'.format(word1,word2,word3,word4))

In [16]:
emd = PreTrainedEmbedding.from_embeddings_file('/content/glove.6B.100d.txt')

Building Index
Finished


In [17]:
emd.compute_and_print_analogy('man', 'he', 'woman')

man : he :: woman : she
man : he :: woman : never
man : he :: woman : her


In [18]:
emd.compute_and_print_analogy('man', 'king', 'woman')

man : king :: woman : queen
man : king :: woman : monarch
man : king :: woman : throne


In [20]:
emd.get_embedding('he')

array([ 0.1225   , -0.058833 ,  0.23658  , -0.28877  , -0.028181 ,
        0.31524  ,  0.070229 ,  0.16447  , -0.027623 ,  0.25214  ,
        0.21174  , -0.059674 ,  0.36133  ,  0.13607  ,  0.18755  ,
       -0.1487   ,  0.31315  ,  0.13368  , -0.59703  , -0.030161 ,
        0.080656 ,  0.26162  , -0.055924 , -0.35351  ,  0.34722  ,
       -0.0055801, -0.57935  , -0.88007  ,  0.42931  , -0.15695  ,
       -0.51256  ,  1.2684   , -0.25228  ,  0.35265  , -0.46419  ,
        0.55648  , -0.57556  ,  0.32574  , -0.21893  , -0.13178  ,
       -1.1027   , -0.039591 ,  0.89643  , -0.9845   , -0.47393  ,
       -0.12855  ,  0.63506  , -0.94888  ,  0.40088  , -0.77542  ,
       -0.35153  , -0.27788  ,  0.68747  ,  1.458    , -0.38474  ,
       -2.8937   , -0.29523  , -0.38836  ,  0.94881  ,  1.3891   ,
        0.054591 ,  0.70486  , -0.65699  ,  0.075648 ,  0.7655   ,
       -0.63365  ,  0.86556  ,  0.42441  ,  0.14796  ,  0.4156   ,
        0.29354  , -0.51295  ,  0.19635  , -0.45568  ,  0.0080

In [21]:
emd.get_embedding('him')

array([ 0.042409 , -0.52195  ,  0.40389  , -0.31683  ,  0.015581 ,
        1.1447   , -0.18197  , -0.061639 , -0.33253  , -0.40223  ,
        0.034373 ,  0.41981  ,  0.031487 , -0.080156 ,  0.22563  ,
       -0.3658   ,  0.18513  ,  0.17092  , -0.6947   ,  0.64301  ,
        0.053546 , -0.45991  , -0.17761  , -0.19595  ,  0.4256   ,
        0.042524 , -0.70944  , -0.92625  ,  1.2065   ,  0.14414  ,
        0.14133  ,  0.68685  ,  0.048357 ,  0.28694  , -0.40552  ,
        0.75591  , -0.79224  , -0.045448 , -0.35752  ,  0.14998  ,
       -0.9691   ,  0.15583  ,  0.7813   , -1.0131   , -0.39037  ,
        0.053001 ,  0.0077983, -0.16409  , -0.055156 , -0.78147  ,
       -0.1504   ,  0.23792  ,  0.29348  ,  1.8954   , -0.44203  ,
       -2.7568   , -0.13436  , -0.30857  ,  1.3768   ,  0.49351  ,
        0.15503  ,  1.206    , -0.51755  ,  0.42908  ,  0.32301  ,
       -0.25093  ,  1.4077   ,  0.99191  , -0.34695  ,  0.4574   ,
       -0.45169  , -0.27574  , -0.040537 , -0.69482  ,  0.1694